# 🇪🇬 Egyptian ID OCR — Notebook 2: Text Mining + Training

This Notebook extracts text from clipped fields and then trains PaddleOCR.

**Input**: `crops_metadata.csv` + `rec/images/`  
**Output**: `crops_labeled.csv` + `rec/train.txt` + trained model

## 1. إعداد البيئة



In [1]:
import sys, os
from pathlib import Path

ROOT = Path(os.getcwd()).parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from IPython.display import display
import matplotlib.pyplot as plt
%matplotlib inline



/home/think/project/egyption_id_ready/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# تحميل metadata
crops_df = pd.read_csv(ROOT / "crops_metadata.csv")
print(f"📋 Total crops: {len(crops_df):,}")
print(f"   Splits: {crops_df['split'].value_counts().to_dict()}")



📋 Total crops: 85,751
   Splits: {'train': 80234, 'valid': 4735, 'test': 782}


## 2. Text extraction — choose the engine


In [3]:
# ═══════════════════════════════════════════════════
#  اختار واحد من الطرق الستة:
# ═══════════════════════════════════════════════════

METHOD = "bakri-airllm"   # "qari" | "qari-airllm" | "gemini" | "bakri" | "bakri-airllm" | "airllm" | "both"
GEMINI_KEY = ""     # ← ضع الـ API key هنا لو هتستخدم Gemini

# AirLLM settings (for 72B models on 4GB GPU)
AIRLLM_MODEL = "Qwen/Qwen2-VL-72B-Instruct"  # or "meta-llama/Llama-3-70B-Instruct"
AIRLLM_CACHE = str(ROOT / "model" / "airllm_cache")
AIRLLM_USE_4BIT = False  # True for <6GB VRAM

# Bakri AirLLM settings (for Bakri OCR on 4GB GPU)
BAKRI_AIRLLM_CACHE = str(ROOT / "model" / "airllm_cache_bakri")
BAKRI_AIRLLM_USE_4BIT = False  # True for <4GB VRAM
BAKRI_AIRLLM_LAYERS_PER_BATCH = 2  # Higher = faster but more VRAM

# QARI AirLLM settings (for QARI-OCR on 4GB GPU)
QARI_AIRLLM_CACHE = str(ROOT / "model" / "airllm_cache_qari")
QARI_AIRLLM_USE_4BIT = False  # True for <3GB VRAM
QARI_AIRLLM_LAYERS_PER_BATCH = 2  # Higher = faster but more VRAM



### Option A: Gemini API



In [4]:
if METHOD in ("gemini", "both"):
    from src.ocr_engines.gemini_ocr import GeminiOCR
    
    gemini = GeminiOCR(api_key=GEMINI_KEY)
    
    # اختبار على عينة
    sample = crops_df[crops_df["split"] == "train"].iloc[0]
    img_path = ROOT / sample["image_path"]
    text = gemini.extract(str(img_path), sample["field"])
    print(f"🔍 Field: {sample['field']}")
    print(f"📝 Text : {text}")



### Option B: QARI-OCR



In [5]:
if METHOD in ("qari", "both"):
    from src.ocr_engines.qari_ocr import QariOCR
    
    qari = QariOCR(use_4bit=False)  # True لو VRAM أقل من 6GB
    
    # اختبار على عينة
    sample = crops_df[crops_df["split"] == "train"].iloc[0]
    img_path = ROOT / sample["image_path"]
    text = qari.extract(str(img_path), sample["field"])
    print(f"🔍 Field: {sample['field']}")
    print(f"📝 Text : {text}")



### Option C: Bakri OCR (Gemma-3-4B)



In [6]:
if METHOD in ("bakri", "both"):
    from src.ocr_engines.bakri_ocr import BakriOCR
    import torch
    import gc
    
    # Clear any leftover GPU memory
    gc.collect()
    torch.cuda.empty_cache()
    
    bakri = BakriOCR(use_4bit=True)  # True لو VRAM أقل من 6GB
    
    # اختبار على عينة
    sample = crops_df[crops_df["split"] == "train"].iloc[0]
    img_path = ROOT / sample["image_path"]
    text = bakri.extract(str(img_path), sample["field"])
    print(f"🔍 Field: {sample['field']}")
    print(f"📝 Text : {text}")



### Option D: Bakri OCR with AirLLM (4GB GPU)

> 🚀 **New:** Run Bakri OCR on 4GB GPU using AirLLM layer-wise inference!  
> ⚠️ **Note:** Slower (~2-5 sec/image) but enables low VRAM usage. Best for offline labeling.


In [7]:
if METHOD in ("bakri-airllm",):
    from src.ocr_engines.bakri_airllm_ocr import BakriAirLLMOCR
    import torch
    import gc
    
    # Clear any leftover GPU memory
    gc.collect()
    torch.cuda.empty_cache()
    
    bakri_airllm = BakriAirLLMOCR(
        model_name="bakrianoo/arabic-legal-documents-ocr-1.0",
        use_4bit=BAKRI_AIRLLM_USE_4BIT,
        cache_dir=BAKRI_AIRLLM_CACHE,
        layers_per_batch=BAKRI_AIRLLM_LAYERS_PER_BATCH,
    )
    
    # اختبار على عينة
    sample = crops_df[crops_df["split"] == "train"].iloc[0]
    img_path = ROOT / sample["image_path"]
    text = bakri_airllm.extract(str(img_path), sample["field"])
    print(f"🔍 Field: {sample['field']}")
    print(f"📝 Text : {text}")



⏳ Loading Bakri OCR bakrianoo/arabic-legal-documents-ocr-1.0 with AirLLM...
ℹ️  Using standard transformers (BetterTransformer deprecated)
   Falling back to standard transformers
   ✅ Standard transformers is fully functional - no action needed


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 883/883 [00:01<00:00, 578.33it/s, Materializing param=model.vision_tower.vision_model.post_layernorm.weight]                      
Some parameters are on the meta device because they were offloaded to the cpu.


✅ Using standard transformers (full precision)


Using `use_fast=True` but `torchvision` is not available. Falling back to the slow image processor.
The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


✅ Processor loaded from cache
✅ Bakri OCR (AirLLM) ready on: cuda
🔍 Field: serial_number
📝 Text : الرجاء توفير النص المراد نسخه من الصورة بدقة عالية بما يكفي لعملية التعرف الضوئي. أنا بحاجة إلى الصورة المكونة للنص وليس مجرد تظليلها.


### Option E: QARI-OCR with AirLLM (4GB GPU)

> 🚀 **New:** Run QARI-OCR on 4GB GPU using AirLLM layer-wise inference!  
> ⚠️ **Note:** Slower (~2-4 sec/image) but enables low VRAM usage. Best for offline labeling.


In [8]:
if METHOD in ("qari-airllm",):
    from src.ocr_engines.qari_airllm_ocr import QariAirLLMOCR
    import torch
    import gc
    
    # Clear any leftover GPU memory
    gc.collect()
    torch.cuda.empty_cache()
    
    qari_airllm = QariAirLLMOCR(
        model_name="NAMAA-Space/Qari-OCR-0.1-VL-2B-Instruct",
        use_4bit=QARI_AIRLLM_USE_4BIT,
        cache_dir=QARI_AIRLLM_CACHE,
        layers_per_batch=QARI_AIRLLM_LAYERS_PER_BATCH,
    )
    
    # اختبار على عينة
    sample = crops_df[crops_df["split"] == "train"].iloc[0]
    img_path = ROOT / sample["image_path"]
    text = qari_airllm.extract(str(img_path), sample["field"])
    print(f"🔍 Field: {sample['field']}")
    print(f"📝 Text : {text}")



### Option F: AirLLM (72B Models on 4GB GPU)

> ⚠️ **Note:** AirLLM uses layer-wise inference — slower but fits 72B models on 4GB GPU.  
> Best for offline labeling with highest accuracy. Expect ~10-30 sec/image.


In [9]:
if METHOD in ("airllm",):
    from src.ocr_engines.airllm_ocr import AirLLMOCR
    import torch
    import gc
    
    # Clear any leftover GPU memory
    gc.collect()
    torch.cuda.empty_cache()
    
    airllm = AirLLMOCR(
        model_name=AIRLLM_MODEL,
        use_4bit=AIRLLM_USE_4BIT,
        cache_dir=AIRLLM_CACHE,
    )
    
    # اختبار على عينة
    sample = crops_df[crops_df["split"] == "train"].iloc[0]
    img_path = ROOT / sample["image_path"]
    text = airllm.extract(str(img_path), sample["field"])
    print(f"🔍 Field: {sample['field']}")
    print(f"📝 Text : {text}")



## 3. Run extraction on all datasets


In [10]:
import time

# Initialize label_text column as strings
if "label_text" not in crops_df.columns:
    crops_df["label_text"] = ""
else:
    # Ensure it's string type
    crops_df["label_text"] = crops_df["label_text"].fillna("").astype(str)

splits_to_label = ["train", "valid"]  # الـ test للتقييم فقط
subset = crops_df[
    crops_df["split"].isin(splits_to_label) &
    (crops_df["label_text"] == "")
]
print(f"📤 Pending: {len(subset):,} crops")



📤 Pending: 84,969 crops


In [11]:
for idx, row in tqdm(subset.iterrows(), total=len(subset), desc="Labeling"):
    img_path = ROOT / row["image_path"]
    if not img_path.exists():
        continue
    
    try:
        if METHOD == "gemini":
            text = gemini.extract(str(img_path), row["field"])
            time.sleep(0.4)  # rate limit
        elif METHOD == "qari":
            text = qari.extract(str(img_path), row["field"])
        elif METHOD == "qari-airllm":
            text = qari_airllm.extract(str(img_path), row["field"])
            time.sleep(0.1)  # small delay for layer-wise inference
        elif METHOD == "bakri":
            text = bakri.extract(str(img_path), row["field"])
        elif METHOD == "airllm":
            text = airllm.extract(str(img_path), row["field"])
            time.sleep(0.1)  # small delay for layer-wise inference
        elif METHOD == "bakri-airllm":
            text = bakri_airllm.extract(str(img_path), row["field"])
            time.sleep(0.1)  # small delay for layer-wise inference
        else:  # both
            text = qari.extract(str(img_path), row["field"])
            crops_df.at[idx, "gemini_text"] = gemini.extract(str(img_path), row["field"])
            time.sleep(0.4)
        
        crops_df.at[idx, "label_text"] = text
    except Exception as e:
        print(f"⚠️ {row['image_path']}: {e}")
    
    # حفظ كل 100 crop
    if idx % 100 == 0:
        crops_df.to_csv(ROOT / "crops_labeled.csv", index=False, encoding="utf-8-sig")

# حفظ نهائي
crops_df.to_csv(ROOT / "crops_labeled.csv", index=False, encoding="utf-8-sig")
# Convert to string to ensure .str accessor works
crops_df["label_text"] = crops_df["label_text"].fillna("").astype(str)
labeled = (crops_df["label_text"].str.len() > 0).sum()
print(f"\n✅ Labeled: {labeled:,} / {len(crops_df):,}")



Labeling:   0%|          | 168/84969 [20:53<175:43:41,  7.46s/it]


KeyboardInterrupt: 

## 4. Examine a sample of the results


In [ ]:
import cv2

labeled_df = crops_df[crops_df["label_text"].str.len() > 0]
samples = labeled_df.sample(min(8, len(labeled_df)), random_state=42)

fig, axes = plt.subplots(2, 4, figsize=(18, 5))
for ax, (_, row) in zip(axes.flat, samples.iterrows()):
    img_path = ROOT / row["image_path"]
    if img_path.exists():
        crop = cv2.imread(str(img_path))
        ax.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
        ax.set_title(f"{row['field']}\n{row['label_text'][:30]}", fontsize=7)
    ax.axis("off")

plt.suptitle("Labeled Crops — Sample", fontsize=12)
plt.tight_layout()
plt.show()



## 5. Prepare PaddleOCR files


In [ ]:
from src.text_cleaner import prepare_paddle_label

crops_df["label_clean"] = crops_df["label_text"].apply(
    lambda x: prepare_paddle_label(str(x)) if pd.notna(x) else ""
)

# بناء القاموس
all_text = "".join(crops_df["label_clean"].dropna().tolist())
unique_chars = sorted(set(c for c in all_text if c.strip() or c == " "))
with open(ROOT / "arabic_dict.txt", "w", encoding="utf-8") as f:
    for c in unique_chars:
        f.write(c + "\n")
print(f"📖 Dictionary: {len(unique_chars)} characters")

# كتابة ملفات التدريب
(ROOT / "rec").mkdir(parents=True, exist_ok=True)
for split, fname in [("train", "train.txt"), ("valid", "val.txt"), ("test", "test.txt")]:
    sub = crops_df[(crops_df["split"] == split) & (crops_df["label_clean"].str.len() > 0)]
    with open(ROOT / "rec" / fname, "w", encoding="utf-8") as f:
        for _, row in sub.iterrows():
            f.write(f"{row['image_path']}\t{row['label_clean']}\n")
    print(f"  {split:6}: {len(sub):>6,} samples → {fname}")



## 6. إحصائيات النصوص



In [ ]:
valid = crops_df[crops_df["label_clean"].str.len() > 0]
lens = valid["label_clean"].str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# توزيع الأطوال
axes[0].hist(lens, bins=40, color="#4CAF50", edgecolor="white")
axes[0].set_title("Text Length Distribution")
axes[0].set_xlabel("Characters")
axes[0].axvline(40, color="red", linestyle="--", label="max_text_length")
axes[0].legend()

# عدد العينات لكل حقل
valid["field"].value_counts().plot(kind="barh", ax=axes[1], color="#2196F3")
axes[1].set_title("Samples per Field")

plt.tight_layout()
plt.show()

print(f"\n📊 Text Length: mean={lens.mean():.1f}, max={lens.max()}, >40={int((lens>40).sum())}")



## 7. Fine-tuning PaddleOCR

> ⚠️ This cell runs training — make sure everything is set up before running.


In [ ]:
# ═══════════════════════════════════════════════════
#  التدريب — PaddleOCR
# ═══════════════════════════════════════════════════

import subprocess

# Download the pre-trained model if it does not existpretrained = ROOT / "arabic_PP-OCRv3_rec_train"
if not pretrained.exists():
    print("⬇️ Downloading pretrained model...")
    subprocess.run([
        "wget", "-q",
        "https://paddleocr.bj.bcebos.com/PP-OCRv3/arabic/arabic_PP-OCRv3_rec_train.tar"
    ], cwd=str(ROOT))
    subprocess.run(["tar", "-xf", "arabic_PP-OCRv3_rec_train.tar"], cwd=str(ROOT))
    print("✅ Downloaded")



In [ ]:
# بدء التدريب
# لاستئناف التدريب من checkpoint، عدّل checkpoints في الـ config:
#   checkpoints: ./output/egyptian_id_rec/latest

print("🏋️ Starting training...")
print("   Config: configs/egyptian_id_rec.yml")
print("   GPU: ✅")
print("   Checkpoint saved every 5 epochs")
print("   Resume: set 'checkpoints' in config to resume\n")

result = subprocess.run(
    ["python", "tools/train.py", "-c", str(ROOT / "configs" / "egyptian_id_rec.yml")],
    cwd=str(ROOT / "PaddleOCR") if (ROOT / "PaddleOCR").exists() else str(ROOT),
)

if result.returncode == 0:
    print("\n✅ Training complete!")
else:
    print(f"\n❌ Training failed (exit code: {result.returncode})")



## 8. Resume training from Checkpoint

If the training stops, run this cell after modifying the config:
```yaml
Global:
  checkpoints: ./output/egyptian_id_rec/latest
```


In [ ]:
# استئناف من آخر checkpoint
CHECKPOINT = ROOT / "output" / "egyptian_id_rec" / "latest"

if CHECKPOINT.exists():
    print(f"▶️ Resuming from: {CHECKPOINT}")
    result = subprocess.run([
        "python", "tools/train.py",
        "-c", str(ROOT / "configs" / "egyptian_id_rec.yml"),
        "-o", f"Global.checkpoints={CHECKPOINT}",
    ], cwd=str(ROOT / "PaddleOCR") if (ROOT / "PaddleOCR").exists() else str(ROOT))
else:
    print(f"❌ No checkpoint found at {CHECKPOINT}")
    print("   Run training first (Cell 7)")



---
✅ **التدريب انتهى!** → شغّل الـ Notebook التالت: `03_evaluate_and_deploy.py`
